# Шаг 4. Очистка данных

Проверяем функции из `app/cleaner.py` на тестовых данных.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

import pandas as pd

from app.cleaner import clean, drop_totals, to_date, to_number
from app.config import SYNTHETIC_DIR
from app.loader import load

cafe = load(SYNTHETIC_DIR / "cafe.csv")["cafe"]
salon = load(SYNTHETIC_DIR / "salon.xlsx")["Sheet1"]

## 1. to_number: текст в число

In [ ]:
# ДО: выручка - текст (тип str), с запятой и пробелами
print(cafe["Выручка, руб"].dtype)
cafe["Выручка, руб"].tail(3)

In [ ]:
# ПОСЛЕ: числа (тип float64). "828 428,10" стало 828428.1
revenue = to_number(cafe["Выручка, руб"])
print(revenue.dtype)
revenue.tail(3)

In [ ]:
# Теперь можно считать (сумма пока включает строку "Итого" - её уберём позже)
print("Сумма:", revenue.sum())
print("Пустых значений:", revenue.isna().sum())

In [ ]:
# Свои примеры - попробуй дописать
examples = pd.Series(["1 200,50", "55,00 руб", "-300", "", "абв", None])
to_number(examples)

### Попробуй сломать: почему нельзя просто pd.to_numeric

In [ ]:
# Обычное превращение в число падает на первой же сумме с запятой
pd.to_numeric(cafe["Выручка, руб"])

In [ ]:
# Сложение текста - не сумма, а склейка строк
"55,00" + "478,40"

## 2. to_date: текст в дату

In [ ]:
# Кафе: короткий год "01.10.24", салон: дата со временем "01.10.2025 09:00"
print("ДО:", cafe["Дата чека"].iloc[0], "|", salon["Дата визита"].iloc[0])
print("ПОСЛЕ:", to_date(cafe["Дата чека"]).iloc[0], "|", to_date(salon["Дата визита"]).iloc[0])
# Строка "Итого" стала пустой датой (NaT) - так мы её потом и найдём
to_date(cafe["Дата чека"]).tail(2)

In [ ]:
# Теперь с датами можно работать: например, взять месяц или день недели
cafe_dates = to_date(cafe["Дата чека"])
print("Первая дата:", cafe_dates.min(), "| последняя:", cafe_dates.max())
cafe_dates.dt.month_name().head(3)

In [ ]:
# Свои примеры - попробуй дописать
examples = pd.Series(["03.04.2025", "31.12.24", "2025-03-04", "2025-03-04 10:30", "Итого", None])
to_date(examples)

### Попробуй сломать: почему нужны настройки dayfirst и ISO

In [ ]:
# Без dayfirst pandas читает "по-американски": 03.04.2025 стало 4 марта, а не 3 апреля
pd.to_datetime(pd.Series(["03.04.2025"]))

In [ ]:
# Ловушка: "2025-03-04" (год-месяц-день) с dayfirst=True превращается в 3 АПРЕЛЯ вместо 4 марта.
# Поэтому to_date разбирает такой формат отдельно
pd.to_datetime(pd.Series(["2025-03-04"]), dayfirst=True, format="mixed")

## 4. clean: всё вместе

По словарю "колонка → роль" сама решает, что превращать в даты, что в числа. Убирает "Итого" и строки без даты. Возвращает чистую таблицу и отчёт.

In [ ]:
# mapping - словарь "колонка -> роль". Для тестовых данных берём его из эталона cafe.json
mapping = json.loads((SYNTHETIC_DIR / "cafe.json").read_text(encoding="utf-8"))
mapping

In [ ]:
# Чистим кафе. Результат - две вещи: чистая таблица и отчёт
cafe_clean, report = clean(cafe, mapping)
print(report)
print()
print(cafe_clean.dtypes)  # даты - datetime, количество и выручка - float64
print()
print("Выручка кафе:", round(cafe_clean["Выручка, руб"].sum(), 2))
cafe_clean.head()

In [ ]:
# Все три файла сразу: у каждого свой mapping (эталон из .json рядом с файлом)
for file_name in ["shop.xlsx", "cafe.csv", "salon.xlsx"]:
    table = list(load(SYNTHETIC_DIR / file_name).values())[0]
    file_mapping = json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))
    _, file_report = clean(table, file_mapping)
    print(file_name, file_report)

In [ ]:
# Вторая линия защиты: итоговая строка с незнакомым названием всё равно уходит - у неё нет даты
test = pd.DataFrame({
    "Дата": ["01.10.2025", "02.10.2025", "Сумма по документу"],
    "Товар": [" Пиво ", "Вода", None],
    "Сумма": ["10,50", "20", "30,50"],
})
test_clean, test_report = clean(test, {"Дата": "date", "Товар": "object", "Сумма": "money"})
print(test_report)
test_clean

## 3. drop_totals: убрать строки "Итого"

In [ ]:
# ДО: последняя строка - "Итого". ПОСЛЕ: её нет, строк на одну меньше
print("Строк до:", len(cafe), "| после:", len(drop_totals(cafe)))
drop_totals(cafe).tail(2)

In [ ]:
# Зачем это нужно: с "Итого" выручка получается ровно в 2 раза больше настоящей
with_total = to_number(cafe["Выручка, руб"]).sum()
without_total = to_number(drop_totals(cafe)["Выручка, руб"]).sum()
print(f"С 'Итого': {with_total:,.2f}")
print(f"Без 'Итого': {without_total:,.2f}")

In [ ]:
# Свои примеры: большие/маленькие буквы и пробелы в начале не мешают
examples = pd.DataFrame({
    "Товар": ["Пиво", "ВСЕГО:", "  итого по группе", "Total", "Всего понемногу", "Сумма по документу"],
    "Сумма": [10, 20, 30, 40, 50, 60],
})
# Заметь: "Всего понемногу" тоже удалилось - известное ограничение.
# А "Сумма по документу" осталась - такого слова нет в стандартном списке
drop_totals(examples)

In [ ]:
# Свой список слов: теперь удаляется "Сумма по документу", а "Всего понемногу" и "Total" остаются
drop_totals(examples, words=("итого", "сумма по документу"))